In [55]:
import pandas as pd
import numpy as np
import pickle
import tensorflow as tf
from tensorflow.keras.models import load_model

In [ ]:
# load the trained model, encoders and scaler
model= load_model('model.keras')

with open('label_encoder_gender.pkl', 'rb') as file:
    label_encoder_gender= pickle.load(file)

with open('onehot_encoder_geo.pkl', 'rb') as file:
    onehot_encoder_geo= pickle.load(file)

with open('scaler.pkl', 'rb') as file:
    scaler= pickle.load(file)

In [57]:
# Example Input data
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [58]:
# convert to dataframe
input_df= pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [59]:
# encode gender column
input_df['Gender']= label_encoder_gender.transform(input_df['Gender'])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [60]:
# encode geography column
geo_encoded_input_data= onehot_encoder_geo.transform(input_df[['Geography']])  #sparse matrix
geo_encoded_df= pd.DataFrame(geo_encoded_input_data.toarray(), columns= onehot_encoder_geo.get_feature_names_out(['Geography']))  #sparse matrix to df
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [61]:
# combine the one hot encoded columns with the original data
input_df= pd.concat([input_df.drop('Geography', axis=1), geo_encoded_df], axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [62]:
# scaling the data
input_df_scaled= scaler.transform(input_df)
input_df_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [63]:
# prediction
prediction= model.predict(input_df_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 346ms/step


array([[0.03784405]], dtype=float32)

In [64]:
prediction_probability= prediction[0][0]
prediction_probability

np.float32(0.03784405)

In [65]:
if prediction_probability>0.5:
    print("The customer is likely to churn")
else:
    print("The customer is unlikely to churn")

The customer is unlikely to churn
